# Final Session: Performance, PySpark & End-to-End Project

**Course:** Python for Data Engineering  
**Covers:** Sessions 13, 14 & 15 (combined)

**What we'll cover:**
- **Part 1** — Performance Optimization (memory, vectorization, chunking)
- **Part 2** — Intro to PySpark (why Spark, Spark vs Pandas, basic transformations)
- **Part 3** — End-to-End Pipeline Project (extract → transform → load)

**Note:** This is a demo-heavy session. Follow along by running each cell. We'll build on everything from the past 12 sessions.

In [ ]:
import pandas as pd
import numpy as np
import time
import sys
import sqlite3
import json
import logging
from datetime import datetime

---

# Part 1: Performance Optimization

When datasets grow from thousands to millions of rows, how you write code matters. Same logic, wildly different speed.

## 1.1 Memory Usage

First rule of large data: know how much memory your DataFrame uses. Pandas loads everything into RAM — if your file is 4 GB and you have 8 GB RAM, you're already in trouble.

In [ ]:
# Create a sample dataset to work with
n_rows = 100_000
np.random.seed(42)

large_df = pd.DataFrame({
    'order_id': range(1, n_rows + 1),
    'customer': np.random.choice(['Alice Johnson', 'Bob Smith', 'Charlie Brown', 'Diana Prince'], n_rows),
    'product': np.random.choice(['Laptop', 'Mouse', 'Keyboard', 'Monitor', 'Headphones'], n_rows),
    'quantity': np.random.randint(1, 20, n_rows),
    'unit_price': np.random.choice([999.99, 29.99, 79.50, 349.99, 59.99], n_rows),
    'region': np.random.choice(['north', 'south', 'east'], n_rows)
})

print(f"Shape: {large_df.shape}")
print(f"\nMemory usage:")
print(large_df.info(memory_usage='deep'))

In [ ]:
# Check memory per column
mem = large_df.memory_usage(deep=True)
print("Memory per column (MB):")
print((mem / 1024 / 1024).round(2))
print(f"\nTotal: {mem.sum() / 1024 / 1024:.2f} MB")

### Optimization: Downcasting & Categories

Two quick wins:
- **Downcast numerics** — `int64` → `int16` if values fit
- **Use `category` dtype** — for columns with few unique values (like region, product)

In [ ]:
# Before optimization
before = large_df.memory_usage(deep=True).sum()

# Optimize
optimized_df = large_df.copy()
optimized_df['quantity'] = pd.to_numeric(optimized_df['quantity'], downcast='integer')
optimized_df['unit_price'] = pd.to_numeric(optimized_df['unit_price'], downcast='float')
optimized_df['customer'] = optimized_df['customer'].astype('category')
optimized_df['product'] = optimized_df['product'].astype('category')
optimized_df['region'] = optimized_df['region'].astype('category')

after = optimized_df.memory_usage(deep=True).sum()

print(f"Before: {before / 1024 / 1024:.2f} MB")
print(f"After:  {after / 1024 / 1024:.2f} MB")
print(f"Saved:  {(1 - after/before) * 100:.1f}%")

In [ ]:
# Compare dtypes
comparison = pd.DataFrame({
    'original': large_df.dtypes,
    'optimized': optimized_df.dtypes
})
print(comparison)

## 1.2 Vectorization vs Loops

This is the single biggest performance lesson: **never loop row-by-row when Pandas can do it in one operation**.

| Approach | Speed | Use when |
|----------|-------|----------|
| `for` loop with `.iterrows()` | Slowest | Almost never |
| `.apply()` with a function | Medium | Custom row logic |
| Vectorized (Pandas/NumPy) | Fastest | Always prefer this |

In [ ]:
# Task: calculate total = quantity * unit_price for each row
df = large_df.copy()

# Method 1: Loop (slow)
start = time.time()
totals_loop = []
for idx, row in df.iterrows():
    totals_loop.append(row['quantity'] * row['unit_price'])
df['total_loop'] = totals_loop
loop_time = time.time() - start

# Method 2: Apply (medium)
start = time.time()
df['total_apply'] = df.apply(lambda row: row['quantity'] * row['unit_price'], axis=1)
apply_time = time.time() - start

# Method 3: Vectorized (fast)
start = time.time()
df['total_vec'] = df['quantity'] * df['unit_price']
vec_time = time.time() - start

print(f"Loop:       {loop_time:.4f}s")
print(f"Apply:      {apply_time:.4f}s")
print(f"Vectorized: {vec_time:.4f}s")
print(f"\nVectorized is {loop_time/vec_time:.0f}x faster than loop")

### Real-world example: conditional logic

Even if-else logic can be vectorized with `np.where` or `np.select`.

In [ ]:
# Tag orders: 'high_value' if total > 500, else 'standard'

# Slow way
start = time.time()
df['tier_loop'] = df.apply(
    lambda row: 'high_value' if row['quantity'] * row['unit_price'] > 500 else 'standard', 
    axis=1
)
apply_time = time.time() - start

# Fast way
start = time.time()
df['tier_vec'] = np.where(df['quantity'] * df['unit_price'] > 500, 'high_value', 'standard')
vec_time = time.time() - start

print(f"Apply: {apply_time:.4f}s")
print(f"np.where: {vec_time:.4f}s")
print(f"\n{apply_time/vec_time:.0f}x faster")

## 1.3 Chunk Processing

When a file is too large to fit in memory, read it in chunks. Process each chunk and combine results.

In [ ]:
# First, create a larger CSV to demo chunking
large_df.to_csv('data/large_orders.csv', index=False)
print(f"Created large_orders.csv with {len(large_df)} rows")

In [ ]:
# Read in chunks of 25,000 rows
chunk_results = []

for i, chunk in enumerate(pd.read_csv('data/large_orders.csv', chunksize=25_000)):
    # Process each chunk: calculate revenue per region
    chunk['total'] = chunk['quantity'] * chunk['unit_price']
    summary = chunk.groupby('region')['total'].sum()
    chunk_results.append(summary)
    print(f"Processed chunk {i+1}: {len(chunk)} rows")

# Combine all chunk results
final = pd.concat(chunk_results).groupby(level=0).sum()
print(f"\nRevenue by region:")
print(final)

In [ ]:
# You can also use chunks to filter and keep only what you need
high_value = []

for chunk in pd.read_csv('data/large_orders.csv', chunksize=25_000):
    chunk['total'] = chunk['quantity'] * chunk['unit_price']
    filtered = chunk[chunk['total'] > 1000]
    high_value.append(filtered)

high_value_df = pd.concat(high_value, ignore_index=True)
print(f"Found {len(high_value_df)} high-value orders out of 100,000")
print(high_value_df.head())

### Performance summary

| Technique | When to use |
|-----------|-------------|
| Downcast dtypes | Always — free memory savings |
| Category dtype | Columns with few unique values |
| Vectorized ops | Any column-wide calculation |
| `np.where` / `np.select` | Conditional column creation |
| Chunk processing | File too large for memory |

---

# Part 2: Intro to PySpark

Pandas works great up to a few million rows on a single machine. Beyond that, you need **distributed computing** — and that's where Apache Spark comes in.

## 2.1 Why Spark?

| | Pandas | PySpark |
|---|---|---|
| **Runs on** | Single machine | Cluster (many machines) |
| **Data size** | Up to ~10 GB | Terabytes+ |
| **Processing** | Eager (runs immediately) | Lazy (builds a plan, runs on action) |
| **Best for** | Exploration, small-medium data | Production pipelines, big data |
| **Memory** | Loads all into RAM | Distributes across nodes |

In real data engineering jobs, you'll often **prototype in Pandas** and **deploy in Spark**. The APIs are intentionally similar.

## 2.2 Setting Up Spark

PySpark needs Java and the `pyspark` package. For this demo, we'll install it locally.

In [ ]:
# Install pyspark (skip if already installed)
# !pip install pyspark

try:
    from pyspark.sql import SparkSession
    from pyspark.sql import functions as F
    
    SPARK_AVAILABLE = True
    print("PySpark imported successfully")
except ImportError:
    SPARK_AVAILABLE = False
    print("PySpark not installed. Install with: pip install pyspark")
    print("You also need Java 8+ installed.")
    print("\nWe'll show the code patterns anyway — the syntax is what matters.")

In [ ]:
# Create a Spark session (this is your entry point to Spark)
if SPARK_AVAILABLE:
    spark = SparkSession.builder \
        .appName("PythonForDE") \
        .master("local[*]") \
        .getOrCreate()
    
    print(f"Spark version: {spark.version}")
    print("Spark UI available at: http://localhost:4040")
else:
    print("Skipping — PySpark not available")

## 2.3 Spark DataFrames — Pandas-like Syntax

Spark DataFrames look similar to Pandas but work differently under the hood.

In [ ]:
if SPARK_AVAILABLE:
    # Read CSV into Spark DataFrame
    spark_df = spark.read.csv('data/orders.csv', header=True, inferSchema=True)
    
    print("Schema:")
    spark_df.printSchema()
    
    print("\nFirst 5 rows:")
    spark_df.show(5)
    
    print(f"Row count: {spark_df.count()}")
else:
    print("""Spark equivalent of Pandas read_csv:
    
    # Pandas
    df = pd.read_csv('data/orders.csv')
    
    # Spark
    df = spark.read.csv('data/orders.csv', header=True, inferSchema=True)
    df.printSchema()   # like df.dtypes
    df.show(5)          # like df.head()
    df.count()          # like len(df)
    """)

### Pandas vs Spark — Side by Side

| Operation | Pandas | PySpark |
|-----------|--------|--------|
| Read CSV | `pd.read_csv('f.csv')` | `spark.read.csv('f.csv', header=True)` |
| Show rows | `df.head()` | `df.show()` |
| Schema | `df.dtypes` | `df.printSchema()` |
| Select cols | `df[['a', 'b']]` | `df.select('a', 'b')` |
| Filter | `df[df['x'] > 5]` | `df.filter(F.col('x') > 5)` |
| New column | `df['new'] = df['a'] * 2` | `df.withColumn('new', F.col('a') * 2)` |
| Group by | `df.groupby('x').sum()` | `df.groupBy('x').sum()` |
| Sort | `df.sort_values('x')` | `df.orderBy('x')` |

In [ ]:
if SPARK_AVAILABLE:
    # Select and filter
    print("Laptops only:")
    spark_df.filter(F.col('product') == 'Laptop') \
            .select('order_id', 'customer', 'quantity') \
            .show()
else:
    print("""# Spark filter + select
    df.filter(F.col('product') == 'Laptop') \\
      .select('order_id', 'customer', 'quantity') \\
      .show()""")

In [ ]:
if SPARK_AVAILABLE:
    # Add a calculated column and aggregate
    result = spark_df.withColumn('total', F.col('quantity') * F.col('unit_price')) \
                     .groupBy('region') \
                     .agg(
                         F.sum('total').alias('revenue'),
                         F.count('order_id').alias('order_count')
                     ) \
                     .orderBy(F.desc('revenue'))
    
    print("Revenue by region:")
    result.show()
else:
    print("""# Spark add column + groupby
    result = df.withColumn('total', F.col('quantity') * F.col('unit_price')) \\
               .groupBy('region') \\
               .agg(
                   F.sum('total').alias('revenue'),
                   F.count('order_id').alias('order_count')
               ) \\
               .orderBy(F.desc('revenue'))
    result.show()""")

In [ ]:
if SPARK_AVAILABLE:
    # Spark also supports SQL directly
    spark_df.createOrReplaceTempView("orders")
    
    sql_result = spark.sql("""
        SELECT region, product, 
               SUM(quantity * unit_price) as revenue
        FROM orders
        GROUP BY region, product
        ORDER BY revenue DESC
        LIMIT 5
    """)
    
    print("Top 5 revenue by region+product (via SQL):")
    sql_result.show()
else:
    print("""# Spark SQL — write SQL directly against DataFrames
    df.createOrReplaceTempView('orders')
    result = spark.sql('SELECT region, SUM(quantity * unit_price) as revenue FROM orders GROUP BY region')
    result.show()""")

## 2.4 Lazy Evaluation

Key concept: Spark doesn't execute anything until you call an **action** (`.show()`, `.count()`, `.collect()`, `.write`). Everything before that is a **transformation** — Spark just builds a plan.

This lets Spark optimize the entire pipeline before running it.

In [ ]:
if SPARK_AVAILABLE:
    # These lines build a plan — nothing executes yet
    plan = spark_df.filter(F.col('quantity') > 2) \
                   .withColumn('total', F.col('quantity') * F.col('unit_price')) \
                   .select('customer', 'product', 'total')
    
    print("Execution plan:")
    plan.explain()  # shows what Spark WILL do
    
    print("\n--- Now actually executing: ---")
    plan.show(5)  # THIS triggers execution
else:
    print("""Spark is lazy — transformations build a plan, actions execute it.
    
    Transformations (lazy): filter, select, withColumn, groupBy, join
    Actions (execute):      show, count, collect, write
    
    plan = df.filter(...).withColumn(...).select(...)  # no execution yet
    plan.explain()  # see the plan
    plan.show()     # NOW it runs""")

In [ ]:
# Clean up Spark session
if SPARK_AVAILABLE:
    spark.stop()
    print("Spark session stopped")

### Spark takeaways

| Concept | Key point |
|---------|----------|
| SparkSession | Entry point — always create one first |
| DataFrame API | Very similar to Pandas — easy to transition |
| Spark SQL | Write plain SQL against DataFrames |
| Lazy evaluation | Transformations build a plan, actions execute it |
| When to use | Data > 10 GB, or production pipelines on clusters |

---

# Part 3: End-to-End Pipeline Project

Let's put everything together. We'll build a complete pipeline that:
1. **Extracts** data from a CSV file
2. **Transforms** it (clean, validate, enrich, aggregate)
3. **Loads** it into a SQLite database
4. Has proper **logging** and **error handling**

This uses skills from every session we've covered.

## 3.1 Pipeline Setup

In [ ]:
# Set up logging (Session 3)
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('data/final_pipeline.log', mode='w'),
        logging.StreamHandler()
    ],
    force=True
)
logger = logging.getLogger('pipeline')
logger.info("Pipeline initialized")

## 3.2 Extract

In [ ]:
def extract(filepath):
    """Read source data. Returns DataFrame or None on failure."""
    try:
        df = pd.read_csv(filepath)
        logger.info(f"Extracted {len(df)} rows from {filepath}")
        logger.info(f"Columns: {list(df.columns)}")
        return df
    except FileNotFoundError:
        logger.error(f"File not found: {filepath}")
        return None
    except Exception as e:
        logger.error(f"Extract failed: {e}")
        return None

# Extract orders
raw_orders = extract('data/orders.csv')
raw_customers = extract('data/customers.csv')

if raw_orders is not None:
    print(raw_orders.head())

## 3.3 Transform

Multiple transform steps, each as a separate function. This is how real pipelines work — small, testable functions chained together.

In [ ]:
def clean_data(df):
    """Clean and standardize raw data."""
    cleaned = df.copy()
    
    # Strip whitespace from string columns (Session 7)
    for col in cleaned.select_dtypes(include='object').columns:
        cleaned[col] = cleaned[col].str.strip()
    
    # Standardize region to lowercase (Session 7)
    cleaned['region'] = cleaned['region'].str.lower()
    
    # Drop duplicates
    before = len(cleaned)
    cleaned = cleaned.drop_duplicates()
    dupes = before - len(cleaned)
    
    logger.info(f"Cleaned data: removed {dupes} duplicates, {len(cleaned)} rows remain")
    return cleaned


def validate_data(df):
    """Validate data quality. Remove bad rows."""
    valid = df.copy()
    
    # Check for nulls
    null_counts = valid.isnull().sum()
    if null_counts.any():
        logger.warning(f"Null values found: {null_counts[null_counts > 0].to_dict()}")
        valid = valid.dropna()
    
    # Quantity must be positive (Session 4 — handling bad data)
    bad_qty = valid[valid['quantity'] <= 0]
    if len(bad_qty) > 0:
        logger.warning(f"Removing {len(bad_qty)} rows with invalid quantity")
        valid = valid[valid['quantity'] > 0]
    
    logger.info(f"Validation passed: {len(valid)} valid rows")
    return valid


def enrich_data(orders_df, customers_df):
    """Join orders with customer info and add calculated columns."""
    # Merge (Session 8)
    enriched = orders_df.merge(customers_df, left_on='customer', right_on='name', how='left')
    
    # Add total column (vectorized — Session 13 optimization)
    enriched['total'] = enriched['quantity'] * enriched['unit_price']
    
    # Add order tier using np.where (vectorized)
    enriched['tier'] = np.where(enriched['total'] > 500, 'premium', 'standard')
    
    # Optimize memory (Session 13)
    enriched['quantity'] = pd.to_numeric(enriched['quantity'], downcast='integer')
    enriched['region'] = enriched['region'].astype('category')
    enriched['tier'] = enriched['tier'].astype('category')
    
    # Select final columns
    enriched = enriched[['order_id', 'customer', 'city', 'product', 'quantity', 
                         'unit_price', 'total', 'region', 'tier']]
    
    logger.info(f"Enriched data: {len(enriched)} rows, memory: {enriched.memory_usage(deep=True).sum() / 1024:.1f} KB")
    return enriched


def build_summary(df):
    """Create aggregated summary tables."""
    # Revenue by region (Session 8 — groupby)
    by_region = df.groupby('region', observed=True).agg(
        total_revenue=('total', 'sum'),
        order_count=('order_id', 'count'),
        avg_order=('total', 'mean')
    ).round(2)
    
    # Top customers
    by_customer = df.groupby('customer').agg(
        total_spent=('total', 'sum'),
        orders=('order_id', 'count')
    ).sort_values('total_spent', ascending=False)
    
    logger.info("Summary tables built")
    return by_region, by_customer

In [ ]:
# Run the transform pipeline
cleaned_orders = clean_data(raw_orders)
valid_orders = validate_data(cleaned_orders)
clean_customers = clean_data(raw_customers)
enriched = enrich_data(valid_orders, clean_customers)
region_summary, customer_summary = build_summary(enriched)

print("\n--- Enriched Orders ---")
print(enriched.head())
print("\n--- Revenue by Region ---")
print(region_summary)
print("\n--- Top Customers ---")
print(customer_summary)

## 3.4 Load

In [ ]:
def load_to_db(enriched_df, region_df, customer_df, db_path='data/final_pipeline.db'):
    """Load DataFrames into SQLite database."""
    try:
        conn = sqlite3.connect(db_path)
        
        # Load each table (Session 12 — pandas to_sql)
        enriched_df.to_sql('orders', conn, if_exists='replace', index=False)
        region_df.to_sql('region_summary', conn, if_exists='replace', index=True)
        customer_df.to_sql('customer_summary', conn, if_exists='replace', index=True)
        
        # Verify the load
        tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
        for table in tables['name']:
            count = pd.read_sql(f"SELECT COUNT(*) as cnt FROM {table}", conn)
            logger.info(f"Loaded table '{table}': {count['cnt'].iloc[0]} rows")
        
        conn.close()
        logger.info(f"All data loaded to {db_path}")
        return True
        
    except Exception as e:
        logger.error(f"Load failed: {e}")
        return False

# Load
success = load_to_db(enriched, region_summary, customer_summary)
print(f"\nPipeline {'succeeded' if success else 'failed'}")

## 3.5 Verify the Output

In [ ]:
# Query the database to verify (Session 12)
conn = sqlite3.connect('data/final_pipeline.db')

print("--- Orders in database ---")
print(pd.read_sql("SELECT * FROM orders LIMIT 5", conn))

print("\n--- Region summary ---")
print(pd.read_sql("SELECT * FROM region_summary", conn))

print("\n--- Premium orders (SQL query) ---")
print(pd.read_sql("""
    SELECT customer, product, total, city 
    FROM orders 
    WHERE tier = 'premium' 
    ORDER BY total DESC
""", conn))

conn.close()

In [ ]:
# Check the log file
print("--- Pipeline Log ---")
with open('data/final_pipeline.log') as f:
    print(f.read())

## 3.6 Full Pipeline as a Single Function

In production, you'd wrap the whole thing in one entry point:

In [ ]:
def run_pipeline(orders_path, customers_path, db_path='data/final_pipeline.db'):
    """Run the complete ETL pipeline."""
    logger.info("=" * 40)
    logger.info("PIPELINE START")
    start = time.time()
    
    try:
        # Extract
        orders = extract(orders_path)
        customers = extract(customers_path)
        if orders is None or customers is None:
            raise ValueError("Extraction failed")
        
        # Transform
        orders = clean_data(orders)
        orders = validate_data(orders)
        customers = clean_data(customers)
        enriched = enrich_data(orders, customers)
        region_summary, customer_summary = build_summary(enriched)
        
        # Load
        success = load_to_db(enriched, region_summary, customer_summary, db_path)
        if not success:
            raise ValueError("Load failed")
        
        elapsed = time.time() - start
        logger.info(f"PIPELINE COMPLETE in {elapsed:.2f}s")
        return True
        
    except Exception as e:
        logger.error(f"PIPELINE FAILED: {e}")
        return False

# Run it
run_pipeline('data/orders.csv', 'data/customers.csv')

---

# Lab: Extend the Pipeline

Modify the pipeline above to add these features:

1. **Add a new transform** — create a `product_summary` table (total revenue and quantity sold per product)
2. **Load it** into the database as a third summary table
3. **Query it** — find the top-selling product by revenue

Use the functions we built as a template.

In [ ]:
# Your code here

---

# Course Summary

| Session | Topic | Key skill |
|---------|-------|----------|
| 1 | Python Basics | Variables, data types, collections |
| 2 | Control Flow & Functions | Loops, functions, lambdas |
| 3 | File Handling & Logging | Read/write CSV/JSON, logging |
| 4 | Exception Handling | try/except, fault-tolerant code |
| 5 | Libraries | pip, venv, package management |
| 6 | Pandas Fundamentals | DataFrames, reading data |
| 7 | Data Cleaning | Nulls, transforms, filtering |
| 8 | Aggregation & Joins | GroupBy, merge, multi-source data |
| 9 | Dates & Time | datetime, time-based features |
| 10 | ETL Pipeline Design | Extract → Transform → Load |
| 11 | Working with APIs | REST, requests, JSON handling |
| 12 | Database Integration | SQL, sqlite3, pandas ↔ DB |
| 13 | Performance | Memory, vectorization, chunks |
| 14 | PySpark Intro | Distributed computing, Spark DataFrames |
| 15 | End-to-End Project | Full pipeline with all skills combined |

### What's next?

To keep growing as a data engineer:
- **Airflow** — schedule and orchestrate pipelines
- **Great Expectations** — automated data quality checks
- **pytest** — write tests for your pipeline functions
- **Cloud platforms** — AWS (Glue, S3, Redshift) or GCP (BigQuery, Dataflow, Cloud Composer)
- **Docker** — containerize your pipelines for deployment